In [ ]:
'''
The main figure program in the paper
'''

In [ ]:
'''
In the third main figure, the calculation procedures for the total scores of each column are shown in ../evaluation/score.ipynb
The results can be found on Figshare: https://doi.org/10.6084/m9.figshare.32064900   data/metric/score
'''

## Figure 4 topo preservation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import LinearSegmentedColormap


method_to_group = {

    "PCA": "LP",
    "SSNMID": "LP",
    "SSNMDI": "LP",
    "pCMF": "LP",
    "tGPLVM": "LP",
    "ZIFA": "LP",
    "scGBM": "LP",
    "GLMPCA": "LP",
    "GLM-PCA": "LP",

    "VAE": "DAE",
    "scGAE": "DAE",
    "scGAE_bn": "DAE",
    "scvis": "DAE",
    "scVis": "DAE",
    "SCDRHA": "DAE",
    "SCDrHA": "DAE",
    "VASC": "DAE",
    "DREAM": "DAE",
    "SAUCIE": "DAE",
    "DRA": "DAE",
    "scScope": "DAE",

    "UMAP": "GRAPH",
    "PHATE": "GRAPH",
    "SIMLR": "GRAPH",
    "EDGE": "GRAPH",
    "SPDR": "GRAPH",

    "TSNE": "METRIC",
    "t-SNE": "METRIC",
    "TriMap": "METRIC",
    "PaCMAP": "METRIC",
    "ivis": "METRIC",
    "SQuaD_MDS": "METRIC",
}

group_order = ["LP", "DAE", "GRAPH", "METRIC"]

# ========= 2. 每个类别的自定义渐变色锚点 =========
group_color_stops = {
    "LP": ["#E8FADD", "#D5F6C0", "#C0F39A"],       # 绿色
    "DAE": ["#DBEFFE", "#BDE2FF", "#94D5FF"],      # 蓝色
    "GRAPH": ["#FFFFD7", "#FFFFB2", "#FFFF84"],    # 黄色
    # "METRIC": ["#E4D8E4", "#A47CA4", "#946794"],   # 紫色
    "METRIC": ["#FFE793", "#FFBC52", "#FF8435"],   # 橙色
    # "METRIC": ["#F3C4D7", "#C7ACE5", "#C7ACE5"],   # 紫色2
    # "METRIC": ["#C0F7B2", "#79EDC2", "#49E7CE"],   # 青色
    # "METRIC": ["#FE9B6B","#FC5945","#E10807"] # 红色
    # "METRIC": ["#43616B","#2C525D","#054760"] # 青色
}


csv_path = "/home/henu/work/result/score/dr3.csv"
metric = "Pearson"

df = pd.read_csv(csv_path)

# 若没有名为 metric 的列，则用第二列
if metric not in df.columns and len(df.columns) >= 2:
    metric = df.columns[1]

# 去掉不需要的方法
df = df[df["Method"] != "ParametricUMAP200"].copy()
df = df[df["Method"] != "ParametricUMAP50"].copy()
df = df[df["Method"] != "SQuaD_MDS"].copy()  # 注意这里你之前是去掉原来的 SQuaD_MDS

# 将 SQuaD_MDS_hybrid 的名字改为 SQuaD_MDS
df["Method"] = df["Method"].replace({"SQuaD_MDS_hybrid": "SQuaD_MDS"})

# 只保留 Method 和 指标列
df = df[["Method", metric]].copy()

# ========= 4. 加入类别信息，并按「类别 -> 指标升序」排序 =========
df["Group"] = df["Method"].map(method_to_group)

# 若有方法没有被映射到任何类别，可以先过滤掉
df = df.dropna(subset=["Group"]).copy()

# 按指定顺序设置 Group 为有序类别
df["Group"] = pd.Categorical(df["Group"],
                             categories=group_order,
                             ordered=True)

# 按类别 + 指标从小到大排序
df = df.sort_values(["Group", metric], ascending=[True, True]).reset_index(drop=True)

# ========= 5. 为每个类别生成“自定义渐变色”，分数越大颜色越深 =========
colors = [None] * len(df)

for g in group_order:
    # 取出属于该组的行（位置索引）
    idx = np.where(df["Group"] == g)[0]
    n = len(idx)
    if n == 0:
        continue

    # 根据 anchor hex 列表生成一个 LinearSegmentedColormap
    stops = group_color_stops[g]
    cmap = LinearSegmentedColormap.from_list(f"cmap_{g}", stops)

    # 当前类内部已经按 metric 升序排序，
    # 用 0.2~1.0 的区间避免太白，数值越大颜色越深
    vals = np.linspace(0.2, 1.0, n)
    for i, pos in enumerate(idx):
        colors[pos] = cmap(vals[i])

# ========= 6. 画图 =========
plt.style.use("default")
plt.figure(figsize=(9, 5))
bars = plt.bar(df["Method"], df[metric], color=colors)

ymax = df[metric].max() if len(df) else 1.0
plt.ylim(0, ymax * 1.15)

plt.ylabel(f"Average {metric}")
plt.gca().yaxis.set_major_locator(MaxNLocator(nbins=6))

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.xticks(rotation=45, ha="right")

# 柱顶标数值
for rect, val in zip(bars, df[metric].values):
    height = rect.get_height()
    plt.text(
        rect.get_x() + rect.get_width() / 2,
        height,
        f"{val:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()

out_svg = "/home/henu/work/other/1/topo/new/Pearson.svg"
plt.savefig(out_svg)
plt.show()

## Figure 5 cluster

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
'''
二维可视化
'''

In [ ]:
embedding = pd.read_csv('/home/henu/work/result/DR/benchmarker/Zhengmix4eq/SSNMDI_2.csv', index_col=0)
meta  = pd.read_csv('/home/henu/work/data/benchmarker/Zhengmix4eq/cell_metadata.csv', index_col=0)
df = embedding.join(meta['cell_type'])
df.columns = ['dim1','dim2','cell_type']

In [ ]:
# —— 1) 锁定类别顺序（可避免颜色随数据顺序变化）——
cats = pd.Index(df['cell_type'].astype(str)).dropna().unique()

# —— 2) 手动颜色（可自行调整/增减；不足时会循环使用）——
manual_colors = [
    "#C0F39A", "#94D5FF", "#FFFF84", "#FF8435"
]
palette = {c: manual_colors[i % len(manual_colors)] for i, c in enumerate(cats)}

# —— 3) 绘图：白色背景、无网格、手动点大小 ——
sns.set_theme(style="white")                      # 去掉seaborn默认网格
fig, ax = plt.subplots(figsize=(7, 5), facecolor="white")

for spine in fig.gca().spines.values():
    spine.set_linewidth(0.5)

ax.set_facecolor("white")

sns.scatterplot(
    data=df,
    x='dim1', y='dim2',
    hue='cell_type',
    palette=palette,      # 使用手动颜色映射
    s=10,                 # 点大小（手动设置）
    linewidth=0,          # 无描边
    ax=ax
)

ax.grid(False)            # 不要背景方格线
ax.set_xlabel('Dim 1')
ax.set_ylabel('Dim 2')
ax.set_title('SSNMDI')
ax.tick_params(axis='both', which='major', labelsize=8)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, title=None)

plt.tight_layout()
plt.savefig('/home/henu/work/other/1/cluster/SSNMDI4.svg',
            dpi=300, bbox_inches='tight', facecolor="white")
plt.show()

In [ ]:
'''
柱状图
'''

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import LinearSegmentedColormap


method_to_group = {

    "PCA": "LP",
    "SSNMID": "LP",
    "SSNMDI": "LP",
    "pCMF": "LP",
    "tGPLVM": "LP",
    "ZIFA": "LP",
    "scGBM": "LP",
    "GLMPCA": "LP",
    "GLM-PCA": "LP",

    "VAE": "DAE",
    "scGAE": "DAE",
    "scGAE_bn": "DAE",
    "scvis": "DAE",
    "scVis": "DAE",
    "SCDRHA": "DAE",
    "SCDrHA": "DAE",
    "VASC": "DAE",
    "DREAM": "DAE",
    "SAUCIE": "DAE",
    "DRA": "DAE",
    "scScope": "DAE",

    "UMAP": "GRAPH",
    "PHATE": "GRAPH",
    "SIMLR": "GRAPH",
    "EDGE": "GRAPH",
    "SPDR": "GRAPH",

    "TSNE": "METRIC",
    "t-SNE": "METRIC",
    "TriMap": "METRIC",
    "PaCMAP": "METRIC",
    "ivis": "METRIC",
    "SQuaD_MDS": "METRIC",
}

group_order = ["LP", "DAE", "GRAPH", "METRIC"]

# ========= 2. 每个类别的自定义渐变色锚点 =========
group_color_stops = {
    "LP": ["#E8FADD", "#D5F6C0", "#C0F39A"],       # 绿色
    "DAE": ["#DBEFFE", "#BDE2FF", "#94D5FF"],      # 蓝色
    "GRAPH": ["#FFFFD7", "#FFFFB2", "#FFFF84"],    # 黄色
    # "METRIC": ["#E4D8E4", "#A47CA4", "#946794"],   # 紫色
    "METRIC": ["#FFE793", "#FFBC52", "#FF8435"],   # 橙色
    # "METRIC": ["#FE9B6B","#FC5945","#E10807"] # 红色
    # "METRIC": ["#43616B","#2C525D","#054760"] # 青色
}


csv_path = "/home/henu/work/result/score/cluster/kmeans/HOMO.csv"
metric = "HOMO"

df = pd.read_csv(csv_path)

# 若没有名为 metric 的列，则用第二列
if metric not in df.columns and len(df.columns) >= 2:
    metric = df.columns[1]

# 去掉不需要的方法
df = df[df["Method"] != "ParametricUMAP200"].copy()
df = df[df["Method"] != "ParametricUMAP50"].copy()
df = df[df["Method"] != "SQuaD_MDS"].copy()  # 注意这里你之前是去掉原来的 SQuaD_MDS

# 将 SQuaD_MDS_hybrid 的名字改为 SQuaD_MDS
df["Method"] = df["Method"].replace({"SQuaD_MDS_hybrid": "SQuaD_MDS"})

# 只保留 Method 和 指标列
df = df[["Method", metric]].copy()

# ========= 4. 加入类别信息，并按「类别 -> 指标升序」排序 =========
df["Group"] = df["Method"].map(method_to_group)

# 若有方法没有被映射到任何类别，可以先过滤掉
df = df.dropna(subset=["Group"]).copy()

# 按指定顺序设置 Group 为有序类别
df["Group"] = pd.Categorical(df["Group"],
                             categories=group_order,
                             ordered=True)

# 按类别 + 指标从小到大排序
df = df.sort_values(["Group", metric], ascending=[True, True]).reset_index(drop=True)

# ========= 5. 为每个类别生成“自定义渐变色”，分数越大颜色越深 =========
colors = [None] * len(df)

for g in group_order:
    # 取出属于该组的行（位置索引）
    idx = np.where(df["Group"] == g)[0]
    n = len(idx)
    if n == 0:
        continue

    # 根据 anchor hex 列表生成一个 LinearSegmentedColormap
    stops = group_color_stops[g]
    cmap = LinearSegmentedColormap.from_list(f"cmap_{g}", stops)

    # 当前类内部已经按 metric 升序排序，
    # 用 0.2~1.0 的区间避免太白，数值越大颜色越深
    vals = np.linspace(0.2, 1.0, n)
    for i, pos in enumerate(idx):
        colors[pos] = cmap(vals[i])

# ========= 6. 画图 =========
plt.style.use("default")
plt.figure(figsize=(9, 5))
bars = plt.bar(df["Method"], df[metric], color=colors)

ymax = df[metric].max() if len(df) else 1.0
plt.ylim(0, ymax * 1.15)

plt.ylabel(f"Average {metric}")
plt.gca().yaxis.set_major_locator(MaxNLocator(nbins=6))

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.xticks(rotation=45, ha="right")

# 柱顶标数值
for rect, val in zip(bars, df[metric].values):
    height = rect.get_height()
    plt.text(
        rect.get_x() + rect.get_width() / 2,
        height,
        f"{val:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()

out_svg = "/home/henu/work/other/1/kmeans/new/HOMO.svg"
plt.savefig(out_svg)
plt.show()

## Figure 6 efficiency

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
datapath = "/home/henu/work/result/efficiency/"
files = [
    os.path.join(datapath,"cell_100.csv"),
    os.path.join(datapath,"cell_500.csv"),
    os.path.join(datapath,"cell_1000.csv"),
    os.path.join(datapath,"cell_2000.csv"),
    os.path.join(datapath,"cell_5000.csv"),
    os.path.join(datapath,"cell_10000.csv"),
    os.path.join(datapath,"cell_20000.csv"),
    os.path.join(datapath,"cell_30000.csv"),
    os.path.join(datapath,"cell_50000.csv"),
    os.path.join(datapath,"cell_73233.csv"),
]
size_re = re.compile(r"cell_(\d+)\.csv$", re.IGNORECASE)

# 1) 读取成长表（dataset_size, method, memory, time）
rows = []
for fp in files:
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file: {fp}")
    m = size_re.search(os.path.basename(fp))
    size = int(m.group(1)) if m else None
    if size is None:
        continue

    df = pd.read_csv(fp)
    # 列名固定：Method, PeakMemory(gb), Time(s)
    need = ["Method", "PeakMemory(gb)", "Time(s)"]
    if not set(need).issubset(df.columns):
        raise ValueError(f"{fp} must contain columns: {need}")

    df = df[need].copy()
    df["Method"] = df["Method"].astype(str).str.strip()

    for _, r in df.iterrows():
        rows.append({
            "dataset_size": size,
            "method": r["Method"],
            "memory": float(r["PeakMemory(gb)"]),
            "time": float(r["Time(s)"]) / 3600.0,
        })

long_df = pd.DataFrame(rows)

# 2) 横轴规模（升序且为 float，与你原代码一致）
cell_counts = np.array(sorted(long_df["dataset_size"].unique()), dtype=float)

# 3) 方法列表（从数据中提取、按字母序稳定）
methods = sorted(long_df["method"].unique().tolist())

# 4) 构造 dict：方法 → 按 cell_counts 顺序排列的列表；缺失填 None
runtime_data, memory_data = {}, {}
for method in methods:
    sub = long_df[long_df["method"] == method]
    time_map = dict(zip(sub["dataset_size"], sub["time"]))
    mem_map  = dict(zip(sub["dataset_size"], sub["memory"]))
    runtime_data[method] = [time_map.get(int(s), None) for s in cell_counts]
    memory_data[method]  = [mem_map.get(int(s), None)  for s in cell_counts]


In [ ]:
# 创建图形窗口
fig, axes = plt.subplots(1, 2, figsize=(20, 10),facecolor="white")

# 图A：运行时间
for method, color in zip(methods, colors):
    axes[0].plot(cell_counts, runtime_data[method], label=method, color=color, marker='o')
axes[0].set_xscale("log")  # 将x轴设为对数坐标  log10(cell number)
axes[0].set_yscale("linear")  # 将y轴设为对数坐标  log10(runtime)
# axes[0].set_title("Runtime vs Number of Cells")
axes[0].set_xlabel("Number of cells")
axes[0].set_ylabel("Runtime (h)")
axes[0].legend()
# axes[0].grid(True, which="both", linestyle="--", linewidth=0.5)  # 网格线，注释掉这一行去掉网格线


# 图B：内存占用
for method, color in zip(methods, colors):
    axes[1].plot(cell_counts, memory_data[method], label=method, color=color, marker='o')
axes[1].set_xscale("log")
axes[1].set_yscale("linear")
# axes[1].set_title("Memory Usage vs Number of Cells")
axes[1].set_xlabel("Number of cells")
axes[1].set_ylabel("Memory (GB)")
axes[1].legend()
# axes[1].grid(True, which="both", linestyle="--", linewidth=0.5)

fig.patch.set_facecolor('white')  # matplotlib背景默认为灰色，改为白色
# 只取掉网格线，背后还是会存在线条，这段代码去除背景所有线条
for ax in axes:
    ax.set_facecolor('white')  # matplotlib背景默认为灰色，改为白色
    ax.grid(False) # 关闭 matplotlib grid

    # 显示 X 和 Y 轴的轴线（只显示左边和底部的轴线）
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)

    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_color("black")
    ax.spines['left'].set_color("black")

    ax.legend(loc="best", fontsize=8, frameon=False, ncol=2)
    # 显示 X 和 Y 轴的轴线 （方形框包裹图）
    # for spine in ax.spines.values():
    #     spine.set_visible(True)
    #     spine.set_linewidth(1)
    #     spine.set_color("black")

# 自动调整布局并保存为 SVG 矢量图
plt.tight_layout()
plt.savefig("/home/henu/work/result/figures/efficiency/legend.svg", format="svg",facecolor="white")
plt.savefig("/home/henu/work/result/figures/efficiency/legend.pdf", format="pdf",facecolor="white")
plt.show()

## Figure 7 stability

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import os

In [ ]:
from matplotlib.colors import LinearSegmentedColormap


colors = [
    "#440154",
    "#3b528b",
    "#21918c",
    "#5ec962",
    "#fde725"
]

cmap = LinearSegmentedColormap.from_list("custom_viridis", colors)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap


colors = [
    "#94D5FF",
    "#BDE2FF",
    "#C0F39A",
    "#D5F6C0",
    "#FFFF84"
]

cmap = LinearSegmentedColormap.from_list("custom_viridis", colors)

In [ ]:
'''
细胞数量
'''
datasets = ['cell_100','cell_500','cell_1k','cell_5k','cell_1w','cell_2w','cell_3w']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/cell/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['100','500','1000','5000','10000','20000','30000']

plt.figure(figsize=(5, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('Cells')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'cell.svg'))
plt.show()

In [ ]:
'''
基因数量
'''
datasets = ['gene_5k','gene_2w','gene_3w','gene_4w','gene_5w']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/gene/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['5000','20000','30000','40000','50000']

plt.figure(figsize=(4, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('Genes')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'gene.svg'))
plt.show()

In [ ]:
'''
细胞类型数量
'''
datasets = ['celltype_7','celltype_9','celltype_11','celltype_13','celltype_15']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/celltype/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['7','9','11','13','15']

plt.figure(figsize=(4, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('Cell Types')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'celltype.svg'))
plt.show()

In [ ]:
'''
批次数量
'''
datasets = ['batch_2','batch_4','batch_6','batch_8','batch_10']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/batch_number/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['2','4','6','8','10']

plt.figure(figsize=(4, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('Batch Number')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'batch_number.svg'))
plt.show()

In [ ]:
'''
批次差异强度
'''
datasets = ['batch_0.2','batch_0.4','batch_0.6','batch_0.8','batch_1.0']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/batch_strength/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['0.2','0.4','0.6','0.8','1.0']

plt.figure(figsize=(4, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('Batch Strength')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'batch_strength.svg'))
plt.show()

In [ ]:
'''
差异基因比例
'''
datasets = ['de_prob_0.05','de_prob_0.15','de_prob_0.2','de_prob_0.25','de_prob_0.3']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/de/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['0.05','0.15','0.2','0.25','0.3']

plt.figure(figsize=(4, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('De_Prob')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'de.svg'))
plt.show()

In [ ]:
'''
差异基因强度
'''
datasets = ['de_0.2','de_0.4','de_0.6','de_0.8','de_1.0']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/de_strength/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['0.2','0.4','0.6','0.8','1.0']

plt.figure(figsize=(4, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('De_Strength')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'de_strength.svg'))
plt.show()

In [ ]:
'''
异常值比例
'''
datasets = ['out_0.1','out_0.2','out_0.3','out_0.4','out_0.5']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/out/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['0.1','0.2','0.3','0.4','0.5']

plt.figure(figsize=(4, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('The proportion of outlier')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'out.svg'))
plt.show()

In [ ]:
'''
dropout概率
'''
datasets = ['dropout_-1','dropout_0', 'dropout_1', 'dropout_2', 'dropout_3']

score_dict = {}

for dataset in datasets:
    score_path = f'/home/henu/work/result/score/stability/1/dropout/{dataset}.csv'
    df = pd.read_csv(score_path)
    s = df.set_index('Method')['Score']
    score_dict[dataset] = s

df_all = pd.concat(score_dict, axis=1)
df_all.columns = datasets

# 去掉不用的方法，并改名
excluded_methods = ['SQuaD_MDS', 'ParametricUMAP200', 'ParametricUMAP50']
df_all = df_all[~df_all.index.isin(excluded_methods)]
df_all.rename(index={'SQuaD_MDS_hybrid': 'SQuadMDS'}, inplace=True)

# ------------ 1. 按类别给方法排序 ------------
categories = {
    "Linear and Probabilistic Factor Models": [
        "PCA", "SSNMDI", "pCMF", "tGPLVM", "ZIFA", "scGBM", "GLMPCA"
    ],
    "Deep Autoencoders and Generative Models": [
        "VAE", "scGAE", "scvis", "SCDRHA", "DREAM","SAUCIE", "DRA", "scScope"
    ],
    "Graph-Based and Diffusion Geometry Models": [
        "UMAP", "PHATE", "SIMLR", "EDGE", "SPDR"
    ],
    "Metric Learning and Structure-Aware Embedding": [
        "TSNE", "TriMap", "PaCMAP", "ivis", "SQuadMDS"
    ]
}

# 按类别直接拼接顺序（严格使用你写的顺序）
method_order = (
    categories["Linear and Probabilistic Factor Models"] +
    categories["Deep Autoencoders and Generative Models"] +
    categories["Graph-Based and Diffusion Geometry Models"] +
    categories["Metric Learning and Structure-Aware Embedding"]
)

df_all = df_all.loc[method_order]
print(df_all.index)

# ------------ 2. 画热图（按你原来的格式） ------------
heatmap_data = df_all.copy()
heatmap_data.columns = ['-1','0','1','2','3']

plt.figure(figsize=(4, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap=cmap,     # 你前面定义好的配色
    linewidths=0.5,
    cbar_kws={'label': 'Score', 'shrink': 0.6, 'extend': 'neither'},
    vmin=0.0,
    vmax=1.0
)

# 坐标轴装饰（照你给的模板）
ax.set_xlabel('Dropout Rate')
ax.set_ylabel('')
# ax.xaxis.set_ticks_position('top')
# ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)
# ax.xaxis.tick_bottom()
plt.xticks(rotation=45)

plt.tight_layout()

save_dir = '/home/henu/work/result/score/stability/1/figures/'
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'dropout.svg'))
plt.show()